<a href="https://colab.research.google.com/github/ga642381/ML2021-Spring/blob/main/HW07/HW07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **作业 7 - Bert（问答任务）**

如有问题，请发送邮件至 ntu-ml-2021spring-ta@googlegroups.com

课件：[链接](https://docs.google.com/presentation/d/1aQoWogAQo_xVJvMQMrGaYiWzuyfO0QyLLAhiMwFyS2w) Kaggle：[链接](https://www.kaggle.com/c/ml2021-spring-hw7) 数据：[链接](https://drive.google.com/uc?id=1znKmX08v9Fygp-dgwo7BKiLIf2qL1FH1)

## 任务描述
- 中文抽取式问答
  - 输入：段落 + 问题
  - 输出：答案

- 目标：学习如何使用 transformers 在下游任务上微调预训练模型

- 待办事项
    - 微调预训练中文 BERT 模型
    - 调整超参数（如 doc_stride）
    - 应用线性学习率衰减
    - 尝试其他预训练模型
    - 改进预处理
    - 改进后处理
- 训练技巧
    - 自动混合精度
    - 梯度累积
    - 模型集成

- 预估训练时间（Tesla T4，启用自动混合精度）
    - Simple: 8 分钟
    - Medium: 8 分钟
    - Strong: 25 分钟
    - Boss: 2 小时

## 安装依赖和导入包

In [ ]:
import json
import numpy as np
import random
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BertForQuestionAnswering, BertTokenizerFast
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import math


device = "cuda" if torch.cuda.is_available() else "cpu"


# 固定随机种子以保证可复现性
def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


same_seeds(0)

In [ ]:
# 启用自动混合精度训练（fp16）
fp16_training = True

if fp16_training:
    from accelerate import Accelerator

    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

## 加载模型和分词器

In [ ]:
model_name = "luhua/chinese_pretrain_mrc_macbert_large"

model = BertForQuestionAnswering.from_pretrained(model_name).to(device)
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# 可以忽略警告信息（因为 QA 的预测头是随机初始化的）

## 读取数据

- 训练集：26935 个问答对
- 验证集：3523 个问答对
- 测试集：3492 个问答对

- {train/dev/test}_questions：
  - 字典列表，包含以下字段：
   - id (int)
   - paragraph_id (int)
   - question_text (string)
   - answer_text (string)
   - answer_start (int)
   - answer_end (int)
- {train/dev/test}_paragraphs：
  - 字符串列表
  - questions 中的 paragraph_id 对应 paragraphs 中的索引
  - 一个段落可能被多个问题引用

In [ ]:
def read_data(file):
    with open(file, "r", encoding="utf-8") as reader:
        data = json.load(reader)
    return data["questions"], data["paragraphs"]


path = "/kaggle/input/competitions/ml2021-spring-hw7/"

train_questions, train_paragraphs = read_data(path + "hw7_train.json")
dev_questions, dev_paragraphs = read_data(path + "hw7_dev.json")
test_questions, test_paragraphs = read_data(path + "hw7_test.json")

## 数据分词

In [ ]:
# 分别对问题和段落进行分词
# add_special_tokens 设为 False，因为在 Dataset 的 __getitem__ 中合并问题和段落时会添加特殊词元

train_questions_tokenized = tokenizer(
    [train_question["question_text"] for train_question in train_questions],
    add_special_tokens=False,
)
dev_questions_tokenized = tokenizer(
    [dev_question["question_text"] for dev_question in dev_questions],
    add_special_tokens=False,
)
test_questions_tokenized = tokenizer(
    [test_question["question_text"] for test_question in test_questions],
    add_special_tokens=False,
)

train_paragraphs_tokenized = tokenizer(train_paragraphs, add_special_tokens=False)
dev_paragraphs_tokenized = tokenizer(dev_paragraphs, add_special_tokens=False)
test_paragraphs_tokenized = tokenizer(test_paragraphs, add_special_tokens=False)

# 可以忽略警告信息，分词后的序列会在 Dataset 的 __getitem__ 中进一步处理后再输入模型

## 数据集和数据加载器

In [ ]:
class QA_Dataset(Dataset):
    def __init__(self, split, questions, tokenized_questions, tokenized_paragraphs):
        self.split = split
        self.questions = questions
        self.tokenized_questions = tokenized_questions
        self.tokenized_paragraphs = tokenized_paragraphs
        self.max_question_len = 40
        self.max_paragraph_len = 150

        ##### TODO: 修改 doc_stride 的值 #####
        self.doc_stride = int(self.max_paragraph_len * 0.25)

        # 输入序列长度 = [CLS] + 问题 + [SEP] + 段落 + [SEP]
        self.max_seq_len = 1 + self.max_question_len + 1 + self.max_paragraph_len + 1

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        tokenized_question = self.tokenized_questions[idx]
        tokenized_paragraph = self.tokenized_paragraphs[question["paragraph_id"]]

        ##### TODO: 预处理 #####
        # 提示：如何防止模型学到不该学的东西

        if self.split == "train":
            # 将答案在段落文本中的起始/结束位置转换为在分词后段落中的起始/结束位置
            answer_start_token = tokenized_paragraph.char_to_token(
                question["answer_start"]
            )
            answer_end_token = tokenized_paragraph.char_to_token(question["answer_end"])

            # 通过截取包含答案的段落部分来获得单个窗口
            mid = (answer_start_token + answer_end_token) // 2
            max_offset = self.max_paragraph_len // 2
            random_offset = np.random.randint(-max_offset, max_offset)
            paragraph_start = max(
                0, min(mid + random_offset - self.max_paragraph_len // 2, len(tokenized_paragraph) - self.max_paragraph_len)
            )
            paragraph_end = paragraph_start + self.max_paragraph_len

            # 截取问题/段落并添加特殊词元（101: CLS, 102: SEP）
            input_ids_question = (
                [101] + tokenized_question.ids[: self.max_question_len] + [102]
            )
            input_ids_paragraph = tokenized_paragraph.ids[
                paragraph_start:paragraph_end
            ] + [102]

            # 将答案在分词段落中的起始/结束位置转换为在窗口中的起始/结束位置
            answer_start_token += len(input_ids_question) - paragraph_start
            answer_end_token += len(input_ids_question) - paragraph_start

            # 填充序列并获取模型输入
            input_ids, token_type_ids, attention_mask = self.padding(
                input_ids_question, input_ids_paragraph
            )
            return (
                torch.tensor(input_ids),
                torch.tensor(token_type_ids),
                torch.tensor(attention_mask),
                answer_start_token,
                answer_end_token,
            )

        # 验证/测试
        else:
            input_ids_list, token_type_ids_list, attention_mask_list = [], [], []

            # 段落被分割为多个窗口，每个窗口的起始位置间隔为 doc_stride
            for i in range(0, len(tokenized_paragraph), self.doc_stride):

                # 截取问题/段落并添加特殊词元（101: CLS, 102: SEP）
                input_ids_question = (
                    [101] + tokenized_question.ids[: self.max_question_len] + [102]
                )
                input_ids_paragraph = tokenized_paragraph.ids[
                    i : i + self.max_paragraph_len
                ] + [102]

                # 填充序列并获取模型输入
                input_ids, token_type_ids, attention_mask = self.padding(
                    input_ids_question, input_ids_paragraph
                )

                input_ids_list.append(input_ids)
                token_type_ids_list.append(token_type_ids)
                attention_mask_list.append(attention_mask)

            return (
                torch.tensor(input_ids_list),
                torch.tensor(token_type_ids_list),
                torch.tensor(attention_mask_list),
            )

    def padding(self, input_ids_question, input_ids_paragraph):
        # 如果序列长度小于 max_seq_len，则填充零
        padding_len = (
            self.max_seq_len - len(input_ids_question) - len(input_ids_paragraph)
        )
        # 输入序列词元在词汇表中的索引
        input_ids = input_ids_question + input_ids_paragraph + [0] * padding_len
        # 段落类型索引，用于区分输入的第一和第二部分，取值为 [0, 1]
        token_type_ids = (
            [0] * len(input_ids_question)
            + [1] * len(input_ids_paragraph)
            + [0] * padding_len
        )
        # 注意力掩码，避免对填充词元执行注意力计算，取值为 [0, 1]
        attention_mask = [1] * (len(input_ids_question) + len(input_ids_paragraph)) + [
            0
        ] * padding_len

        return input_ids, token_type_ids, attention_mask


train_set = QA_Dataset(
    "train", train_questions, train_questions_tokenized, train_paragraphs_tokenized
)
dev_set = QA_Dataset(
    "dev", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized
)
test_set = QA_Dataset(
    "test", test_questions, test_questions_tokenized, test_paragraphs_tokenized
)

train_batch_size = 4
gradient_accumulation_steps = 8

# 注意：不要修改 dev_loader / test_loader 的 batch_size！
# 虽然 batch_size=1，但实际上是一个由同一问答对的多个窗口组成的批次
train_loader = DataLoader(
    train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True
)
dev_loader = DataLoader(dev_set, batch_size=1, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, pin_memory=True)

## 评估函数

In [ ]:
def evaluate(data, output):
    ##### TODO: 后处理 #####
    # 后处理中存在 bug，且有改进空间
    # 提示：打开预测文件看看哪里出了问题

    answer = ""
    max_prob = float("-inf")
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        # 通过选择概率最大的起始位置/结束位置来获取答案
        start_prob, start_index = torch.max(
            torch.softmax(output.start_logits[k], dim=0), dim=0
        )
        end_prob, end_index = torch.max(
            torch.softmax(output.end_logits[k], dim=0), dim=0
        )

        # 确保起始位置不大于结束位置，过滤无效预测
        if start_index <= end_index:
            prob = start_prob * end_prob

            # 如果计算出的概率大于之前窗口的结果，则替换答案
            if prob > max_prob:
                max_prob = prob
                # 将词元转换为字符（例如 [1920, 7032] --> "大 金"）
                answer = tokenizer.decode(data[0][0][k][start_index : end_index + 1])
        else:
            continue

    # 去除答案中的空格（例如 "大 金" --> "大金"）
    return answer.replace(" ", "")

## 训练

In [ ]:
logging_step = 100
validation = True
num_epoch = 3
learning_rate = 5e-5
optimizer = AdamW(model.parameters(), lr=learning_rate)

warmup_steps = 200
total_steps = math.ceil(len(train_set) / (train_batch_size * gradient_accumulation_steps)) * num_epoch

warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=total_steps - warmup_steps, eta_min=1e-6
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps],
)

if fp16_training:
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

model.train()
best_acc = 0

print("开始训练 ...")

for epoch in range(num_epoch):
    step = 0
    train_loss = train_acc = 0
    actual_logging_steps = 0

    for batch_idx, data in enumerate(tqdm(train_loader)):
        # 将所有数据加载到 GPU
        data = [i.to(device) for i in data]

        output = model(
            input_ids=data[0],
            token_type_ids=data[1],
            attention_mask=data[2],
            start_positions=data[3],
            end_positions=data[4],
        )

        # 选择概率最大的起始位置/结束位置
        start_index = torch.argmax(output.start_logits, dim=1)
        end_index = torch.argmax(output.end_logits, dim=1)

        # 只有当 start_index 和 end_index 都正确时，预测才算正确
        train_acc += ((start_index == data[3]) & (end_index == data[4])).float().mean()
        # 梯度累积：loss 除以累积步数
        loss = output.loss / gradient_accumulation_steps
        train_loss += loss.item()
        actual_logging_steps += 1

        if fp16_training:
            accelerator.backward(loss)
        else:
            loss.backward()

        # 每 gradient_accumulation_steps 步更新一次权重
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

            # 每隔 logging_step 打印训练损失和准确率
            if step % logging_step == 0:
                avg_loss = train_loss / actual_logging_steps * gradient_accumulation_steps
                avg_acc = train_acc / actual_logging_steps
                print(
                    f"Epoch {epoch + 1} | Step {step} | loss = {avg_loss:.3f}, acc = {avg_acc:.3f}"
                )
                train_loss = train_acc = 0
                actual_logging_steps = 0

    if validation:
        print("评估验证集 ...")
        model.eval()
        with torch.no_grad():
            dev_acc = 0
            for i, data in enumerate(tqdm(dev_loader)):
                output = model(
                    input_ids=data[0].squeeze(dim=0).to(device),
                    token_type_ids=data[1].squeeze(dim=0).to(device),
                    attention_mask=data[2].squeeze(dim=0).to(device),
                )
                # 只有当答案文本完全匹配时，预测才算正确
                dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
            dev_acc = dev_acc / len(dev_loader)
            print(f"验证 | Epoch {epoch + 1} | acc = {dev_acc:.3f}")
            if dev_acc > best_acc:
                best_acc = dev_acc
                model.save_pretrained("saved_model")
                print(f"新最优！保存模型, acc = {best_acc:.3f}")
        model.train()

if not validation:
    print("保存模型 ...")
    model.save_pretrained("saved_model")
else:
    print(f"训练结束，最优验证 acc = {best_acc:.3f}")

## 测试

In [ ]:
print("评估测试集 ...")

result = []

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader):
        output = model(
            input_ids=data[0].squeeze(dim=0).to(device),
            token_type_ids=data[1].squeeze(dim=0).to(device),
            attention_mask=data[2].squeeze(dim=0).to(device),
        )
        result.append(evaluate(data, output))

result_file = "result.csv"
with open(result_file, "w") as f:
    f.write("ID,Answer\n")
    for i, test_question in enumerate(test_questions):
        # 将答案中的逗号替换为空字符串（因为 csv 以逗号分隔）
        # Kaggle 上的答案也做了同样的处理
        f.write(f"{test_question['id']},{result[i].replace(',','')}\n")

print(f"完成！结果保存在 {result_file}")